# Figure 4 — Organocatalyst chemical space 

Publication-ready t-SNE map of the organocatalyst chemical space, coloured by the
rule-based functional-group class (`fg_class`).

**Design decisions baked into this notebook (fixed for reproducibility):**

- **Deduplication keeps atropisomers separate.** Rows with the same 2D SMILES but a
  different `axial_configuration` are treated as distinct entries (key =
  `canonical_smiles` + `axial_configuration`).
- **Fingerprint is left as plain Morgan (no axial chirality).** Consequently
  atropisomers share an identical fingerprint and **coincide in the 2D projection** —
  this is stated in the figure caption rather than worked around.
- **Single label column:** `fg_class` only.
- **Legend is placed outside the plot** (right side) so it never covers the points.
- **t-SNE parameters are fixed** (`perplexity`, `random_state`) so the figure is
  reproducible run-to-run.

Install once:
`pip install rdkit scikit-learn scipy matplotlib pandas`

In [ ]:
import pandas as pd

# --- EDIT THIS PATH to your dataset ---
DATA_PATH = "Mannich_dataset.csv"
df = pd.read_excel(DATA_PATH)
print(f"rows: {len(df)}")

In [ ]:
import os, tempfile
import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger
from rdkit.Chem import DataStructs, Draw

# Plain Morgan fingerprint (radius=2, 2048 bits), NO chirality encoding.
# Atropisomers (same 2D SMILES) therefore map to identical fingerprints.
try:
    from rdkit.Chem import rdFingerprintGenerator
    _MFPGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    _morgan = lambda m: _MFPGEN.GetFingerprint(m)
except Exception:
    from rdkit.Chem import AllChem
    _morgan = lambda m: AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048)

RDLogger.DisableLog("rdApp.*")

# Only the rule-based functional-group class is kept as a label column.
LABEL_COLS = ["fg_class"]

## 1. Parse SMILES and deduplicate
Key = canonical SMILES **+ `axial_configuration`** (when that column exists), so
atropisomers are kept as separate entries.

In [ ]:
def _canon(smiles):
    if not isinstance(smiles, str):
        return None
    m = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(m) if m is not None else None


def prepare_unique(df, smiles_col="organocatalyst", config_col="axial_configuration",
                   keep_stereo=True):
    """Collapse duplicates to unique structures. Same SMILES with different
    axial_configuration are kept as SEPARATE rows. Returns df_unique with a
    `count` column, parsed `mol`, and `canonical_smiles`."""
    work = df.copy()
    work["canonical_smiles"] = work[smiles_col].apply(_canon)
    bad = work[work["canonical_smiles"].isnull()]
    if len(bad):
        print(f"[!] not parsed as SMILES: {len(bad)} rows")
    work = work[work["canonical_smiles"].notnull()]
    if not keep_stereo:
        work["canonical_smiles"] = work["canonical_smiles"].apply(
            lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False))

    key = ["canonical_smiles"]
    if config_col in work.columns:
        key.append(config_col)
        print(f"[dedup] key = {key} (same SMILES + different {config_col} kept separately)")
    counts = work.groupby(key).size().rename("count")
    uniq = work.drop_duplicates(key).merge(counts, on=key).reset_index(drop=True)
    uniq = uniq.rename(columns={smiles_col: "example_input"})
    uniq["mol"] = uniq["canonical_smiles"].apply(Chem.MolFromSmiles)
    print(f"[dedup] input: {len(df)}; valid: {len(work)}; unique: {len(uniq)}")
    return uniq

## 2. Fingerprints and Tanimoto distance

In [ ]:
def compute_fingerprints(mols):
    fps = [_morgan(m) for m in mols]
    arr = np.zeros((len(fps), 2048), dtype=np.int8)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return fps, arr


def tanimoto_distance_square(fps):
    n = len(fps)
    dist = np.zeros((n, n))
    for i in range(n):
        dist[i] = 1.0 - np.array(DataStructs.BulkTanimotoSimilarity(fps[i], fps))
    np.fill_diagonal(dist, 0.0)
    return (dist + dist.T) / 2.0

## 3. Functional-group class (`fg_class`)

Single label by **priority** (first match wins). `squaramide` and the merged
`urea_thiourea_guanidine_isothiourea` outrank scaffold motifs.
`proline_pyrrolidine_derivatives` (pyrrolidine + thiazolidine/thioproline) is placed
above `1,2-diamine` and `amino_acid`, and suppresses them, so proline-derived diamines
and thioproline land in the proline group. `motifs` keeps all raw matches for transparency.

In [ ]:
FG_SMARTS = {
    "squaramide":                          "[#7]~[#6]1~[#6](~[#7])~[#6](=O)~[#6]1=O",
    "urea_thiourea_guanidine_isothiourea": ["[NX3][CX3](=[OX1,SX1,NX2,NX3+])[NX3]",
                                            "[NX3][CX3](=[NX2])[SX2]"],
    "phosphoric_acid":                     "[PX4](=O)([OX2H1,OX1-,NX3])[OX2]",
    "cinchona_quinuclidine":               "N1(CC2)CCC2CC1",
    "imidazolidinone":                     "O=C1NCNC1",
    "proline_pyrrolidine_derivatives":     ["[NX3;R]1CCCC1",
                                            "[#7;R]1[#6][#16;R][#6][#6]1"],
    "1,2-diamine":                         "[#7][CX4][CX4][#7]",
    "amino_acid":                          "[NX3,NX4+;!$([NX3]C=[O,S,N])][CX4][CX3](=O)[OX2H1,OX1-,$([OX2][#6]),NX3]",
    "ammonium_betaine":                    "[NX4+]",
    "binaphthyl_azepine":                  "[c;r6]1[c;r6][CH2;r7][NX3;r7][CH2;r7][c;r6][c;r6]1",
    "binaphthyl":                          "c1ccc2ccccc2c1-c1ccc2ccccc2c1",
}
_compile = lambda v: [Chem.MolFromSmarts(p) for p in (v if isinstance(v, list) else [v])]
FG_COMPILED = [(k, _compile(v)) for k, v in FG_SMARTS.items()]
_PROL = "proline_pyrrolidine_derivatives"


def _hits(mol):
    s = {k for k, ps in FG_COMPILED if any(p is not None and mol.HasSubstructMatch(p) for p in ps)}
    if _PROL in s:                   s.discard("amino_acid"); s.discard("1,2-diamine")
    if "imidazolidinone" in s:       s.discard("amino_acid")
    if "cinchona_quinuclidine" in s: s.discard("1,2-diamine")
    return s


def assign_fg_class(mols):
    order = list(FG_SMARTS)
    return pd.Series([next((k for k in order if k in _hits(m)), "other") for m in mols])


def all_motifs(mols):
    return pd.Series(["|".join(sorted(_hits(m))) or "other" for m in mols])

## 4. Helpers: collapse small classes, map labels back to original df

In [ ]:
def collapse_small_classes(labels, min_size=5, other_label="other", protect=None):
    """Any class smaller than min_size -> other."""
    labels = pd.Series(labels).astype(object).reset_index(drop=True)
    protect = set(protect or [])
    sizes = labels.value_counts()
    small = set(sizes[sizes < min_size].index) - {other_label} - protect
    return labels.where(~labels.isin(small), other_label)


def map_clusters_to_original(df_original, df_unique, smiles_col="organocatalyst",
                             config_col="axial_configuration", keep_stereo=True):
    """Attach fg_class back to every original row (merged on SMILES,
    plus axial_configuration when present)."""
    out = df_original.copy()
    out["canonical_smiles"] = out[smiles_col].apply(_canon)
    if not keep_stereo:
        out["canonical_smiles"] = out["canonical_smiles"].apply(
            lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)
            if isinstance(s, str) else None)
    key = ["canonical_smiles"]
    if config_col in df_unique.columns and config_col in out.columns:
        key.append(config_col)
    cols = key + [c for c in LABEL_COLS if c in df_unique.columns]
    return out.merge(df_unique[cols], on=key, how="left")

## 5. Runner: deduplicate, fingerprint, assign `fg_class`

In [ ]:
def run_all(df, smiles_col="organocatalyst", config_col="axial_configuration",
            keep_stereo=True, min_size_collapse=5):
    d = prepare_unique(df, smiles_col, config_col, keep_stereo)
    mols = d["mol"].tolist()
    fps, arr = compute_fingerprints(mols)
    dist = tanimoto_distance_square(fps)

    d["fg_class"] = assign_fg_class(mols).values
    d["motifs"]   = all_motifs(mols).values
    if min_size_collapse:
        d["fg_class"] = collapse_small_classes(d["fg_class"], min_size_collapse).values

    d.attrs["dist"] = dist
    d.attrs["fp_array"] = arr
    return d


def summarize(d):
    for c in LABEL_COLS:
        if c in d.columns:
            print(f"{c}: {d[c].nunique()} groups -> {d[c].value_counts().to_dict()}")

## 6. t-SNE plot

**Fixed for the final figure.** t-SNE on the precomputed Tanimoto distance matrix,
coloured by `fg_class`, with a fixed `random_state` for reproducibility.

- `perplexity=30` — balances local vs. global structure; lower (5–15) emphasises
  tight local clusters, higher (40–50) spreads them out. Must be < n_samples.
- `metric="precomputed"` — uses the same Morgan/Tanimoto distances as before.
- Legend outside on the right.

> **Note on atropisomers:** they are kept as separate dataset entries but share an
> identical Morgan fingerprint (axial chirality is not encoded), so they overlap in the
> projection. State this in the caption.

In [ ]:
NICE_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860",
               "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD", "#A1C9F4", "#FFB482",
               "#8DE5A1", "#FF9F9B", "#D0BBFF", "#DEBB9B", "#FAB0E4", "#CFCFCF",
               "#FFFEA3", "#B9F2F0"]


def _safe_save(fig, out_dir, filename):
    path = os.path.join(out_dir or os.getcwd(), filename)
    try:
        fig.savefig(path, dpi=300, bbox_inches="tight"); print(f"[plot] {path}")
    except OSError:
        path = os.path.join(tempfile.gettempdir(), filename)
        fig.savefig(path, dpi=300, bbox_inches="tight"); print(f"[plot] read-only cwd -> {path}")


def plot_tsne(d, color_by="fg_class", out_dir=None, filename=None,
              perplexity=30.0, random_state=0, max_iter=1000,
              save_formats=("png", "pdf", "svg")):
    """Final t-SNE figure on the precomputed Tanimoto distance matrix.

      perplexity : balances local vs. global structure (typical 5-50);
                   must be < number of samples. Lower -> tighter local clusters.
    Fixed random_state makes the figure reproducible.
    """
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    from sklearn.manifold import TSNE

    # publication style
    mpl.rcParams.update({
        "font.family": "Arial", "pdf.fonttype": 42, "ps.fonttype": 42,
        "axes.spines.top": False, "axes.spines.right": False,
    })

    dist = d.attrs.get("dist")
    if dist is None:
        fps, _ = compute_fingerprints(d["mol"].tolist())
        dist = tanimoto_distance_square(fps)
    n = dist.shape[0]
    perp = min(perplexity, max(5.0, (n - 1) / 3.0))   # keep perplexity < n_samples

    # sklearn renamed n_iter -> max_iter in 1.5; support both transparently
    import inspect
    tsne_kw = dict(n_components=2, metric="precomputed", init="random",
                   perplexity=perp, random_state=random_state)
    if "max_iter" in inspect.signature(TSNE).parameters:
        tsne_kw["max_iter"] = max_iter
    else:
        tsne_kw["n_iter"] = max_iter
    coords = TSNE(**tsne_kw).fit_transform(dist)

    cats = sorted(d[color_by].astype(str).unique())
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, c in enumerate(cats):
        mask = d[color_by].astype(str).values == c
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   color=NICE_COLORS[i % len(NICE_COLORS)],
                   s=60, edgecolor="white", linewidth=0.5,
                   label=f"{c} (n={int(mask.sum())})")
    ax.set_xlabel("t-SNE-1"); ax.set_ylabel("t-SNE-2")
    # legend inside the plot, lower-left corner
    ax.legend(loc="lower left", fontsize=8, framealpha=0.7, labelspacing=0.3,
              handletextpad=0.3, borderpad=0.4)
    fig.tight_layout()

    base = filename or f"tsne_{color_by}"
    for ext in save_formats:
        _safe_save(fig, out_dir, f"{base}.{ext}")
    plt.show()
    return coords

## 7. Build the space and draw the figure

In [ ]:
d = run_all(df, smiles_col="organocatalyst", min_size_collapse=5)
summarize(d)

In [ ]:
coords = plot_tsne(d, color_by="fg_class", out_dir=".")

## 8. Write `organocatalyst_class` back to the dataset and save

Maps `fg_class` onto every original row (matched on canonical SMILES **+ `axial_configuration`**) and stores it in the column `organocatalyst_class`. The enriched table is written next to the source file as `*_classified.xlsx`.

In [ ]:
# Map fg_class onto every original row (incl. duplicates / axial variants),
# matched on canonical SMILES + axial_configuration, and store it in
# the column `organocatalyst_class`.
full = map_clusters_to_original(df, d)               # adds canonical_smiles + fg_class
full["organocatalyst_class"] = full["fg_class"]      # write into the requested column

# write the value straight into the working df as well
df["organocatalyst_class"] = full["organocatalyst_class"].values

# sanity checks
n_missing = df["organocatalyst_class"].isnull().sum()
print(f"rows: {len(df)} | classified: {len(df) - n_missing} | unmatched: {n_missing}")
print(df["organocatalyst_class"].value_counts(dropna=False).to_dict())
df[["organocatalyst", "organocatalyst_class"]].head()

### Save to xlsx

In [ ]:
# Save the enriched dataset next to the source file.
out_path = os.path.splitext(DATA_PATH)[0] + "_classified.xlsx"
try:
    df.to_excel(out_path, index=False)
    print(f"[saved] {out_path}")
except OSError:
    out_path = os.path.join(tempfile.gettempdir(),
                            os.path.basename(os.path.splitext(DATA_PATH)[0]) + "_classified.xlsx")
    df.to_excel(out_path, index=False)
    print(f"[saved, read-only cwd] {out_path}")

## 9. Composite figure (A: t-SNE, B: class histogram)

Single figure with two panels sharing one colour scheme:

- **Panel A** — t-SNE map coloured by `fg_class`.
- **Panel B** — reaction-occurrence histogram per class (rows of the dataset),
  each bar coloured to match its t-SNE cluster.

Colours come from the **same** rule used inside `plot_tsne`
(`NICE_COLORS[i]` over `sorted(d["fg_class"].unique())`), so a class has an identical
colour in both panels. Class names are shown with human-readable labels.

In [ ]:
# Pretty, human-readable class labels (raw fg_class -> display name)
CLASS_LABELS = {
    "urea_thiourea_guanidine_isothiourea": "ureas, thioureas,\nguanidines, isothioureas",
    "proline_pyrrolidine_derivatives":     "proline and\npyrrolidine derivatives",
    "squaramide":                          "squaramides",
    "cinchona_quinuclidine":               "cinchona quinuclidines",
    "amino_acid":                          "amino acids",
    "other":                               "other",
    "binaphthyl_azepine":                  "binaphthyl azepines",
    "phosphoric_acid":                     "phosphoric acids",
    "1,2-diamine":                         "1,2-diamines",
    "ammonium_betaine":                    "ammonium betaines",
    "imidazolidinone":                     "imidazolidinones",
}
pretty = lambda c: CLASS_LABELS.get(c, c.replace("_", " "))


def plot_composite(d, df, color_by="fg_class", class_col="organocatalyst_class",
                   perplexity=30.0, random_state=0, max_iter=1000,
                   out_dir=None, filename=None, save_formats=("png", "pdf", "svg")):
    """Composite A/B figure: t-SNE (A) + reaction-occurrence histogram (B),
    sharing one colour scheme keyed on fg_class."""
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import numpy as np
    from sklearn.manifold import TSNE
    import inspect

    mpl.rcParams.update({
        "font.family": "Arial", "pdf.fonttype": 42, "ps.fonttype": 42,
        "axes.spines.top": False, "axes.spines.right": False,
    })

    # --- shared colour map: identical rule to plot_tsne ---
    cats = sorted(d[color_by].astype(str).unique())
    cmap = {c: NICE_COLORS[i % len(NICE_COLORS)] for i, c in enumerate(cats)}

    # --- t-SNE coords (same params as plot_tsne) ---
    dist = d.attrs.get("dist")
    if dist is None:
        fps, _ = compute_fingerprints(d["mol"].tolist())
        dist = tanimoto_distance_square(fps)
    n = dist.shape[0]
    perp = min(perplexity, max(5.0, (n - 1) / 3.0))
    tsne_kw = dict(n_components=2, metric="precomputed", init="random",
                   perplexity=perp, random_state=random_state)
    if "max_iter" in inspect.signature(TSNE).parameters:
        tsne_kw["max_iter"] = max_iter
    else:
        tsne_kw["n_iter"] = max_iter
    coords = TSNE(**tsne_kw).fit_transform(dist)

    fig, (axA, axB) = plt.subplots(2, 1, figsize=(11, 13),
                                   gridspec_kw={"height_ratios": [1.25, 1], "hspace": 0.32})

    # ---------- Panel A: t-SNE ----------
    for c in cats:
        mask = d[color_by].astype(str).values == c
        axA.scatter(coords[mask, 0], coords[mask, 1], color=cmap[c],
                    s=60, edgecolor="white", linewidth=0.5,
                    label=f"{pretty(c)} (n={int(mask.sum())})")
    axA.set_xlabel("t-SNE-1"); axA.set_ylabel("t-SNE-2")
    axA.legend(loc="lower left", fontsize=7.5, framealpha=0.7, labelspacing=0.3,
               handletextpad=0.3, borderpad=0.4)

    # ---------- Panel B: reaction-occurrence histogram ----------
    reactions = df[class_col].value_counts()
    classes = list(reactions.sort_values(ascending=False).index)
    react_vals = [int(reactions.get(c, 0)) for c in classes]
    x = np.arange(len(classes))
    bar_colors = [cmap.get(c, "#8C8C8C") for c in classes]

    axB.bar(x, react_vals, width=0.8, color=bar_colors, alpha=0.9,
            edgecolor="white", linewidth=0.6)
    for xi, v in zip(x, react_vals):
        if v: axB.text(xi, v, str(v), ha="center", va="bottom", fontsize=8, color="#333333")
    axB.set_xticks(x)
    axB.set_xticklabels([pretty(c) for c in classes], rotation=40, ha="right", fontsize=8.5)
    axB.set_ylabel("Reaction occurrences (dataset rows)")
    axB.set_xlabel("Organocatalyst class")
    axB.set_ylim(0, max(react_vals) * 1.12 if react_vals else 1)

    # ---------- panel letters A / B (top-left of each) ----------
    for ax, letter in ((axA, "A"), (axB, "B")):
        ax.text(-0.06, 1.02, letter, transform=ax.transAxes,
                fontsize=18, fontweight="bold", va="bottom", ha="right")

    base = filename or "composite_tsne_histogram"
    for ext in save_formats:
        path = os.path.join(out_dir or os.getcwd(), f"{base}.{ext}")
        try:
            fig.savefig(path, dpi=300, bbox_inches="tight"); print(f"[plot] {path}")
        except OSError:
            path = os.path.join(tempfile.gettempdir(), f"{base}.{ext}")
            fig.savefig(path, dpi=300, bbox_inches="tight"); print(f"[plot] read-only cwd -> {path}")
    plt.show()
    return coords, reactions


coords, reactions = plot_composite(d, df, out_dir=".")